# PlastPrior: source data inspection

This notebook presents the first stage of the PlastPrior workflow. It describes the local input collection; chemical identity resolution and scoring have not yet been implemented.

## Setup

Use the **PlastPrior (Python 3.12)** kernel, restart it, and run all cells from the project directory. The `README.md` describes installation and the inspection scope. Inputs are read without modification; generated reports are written to `results/`.

In [ ]:
from pathlib import Path
from IPython.display import display

if not (Path.cwd() / "plastprior" / "inputs.py").is_file():
    raise RuntimeError("Run this notebook from the PlastPrior project directory.")

from plastprior.inputs import inspect_inputs

DATA_ROOT = Path("data")
REPORT_DIR = Path("results")

## Stage 1 — Input validation and ingestion

Inspect the declared source files and selected worksheets. Versions and snapshot dates are recorded metadata; **Not recorded** indicates a provenance gap. Parsed rows are source records, not unique chemicals, and must not be summed across overlapping inventories. Table roles distinguish inventory records, identity references, and literature links.

Stage 1 requires the source-file and table checks to pass. Auxiliary inputs are listed separately and receive **presence checks only**; their contents must be validated by the stages that use them.

In [ ]:
inspection = inspect_inputs(DATA_ROOT)
inspection.write_reports(REPORT_DIR)

display(inspection.sources[[
    "source", "filename", "version", "snapshot_date", "status"
]])
display(inspection.tables[[
    "source_id", "table_id", "table_role", "parsed_rows", "nonempty_rows", "columns", "status"
]])

**Review the inspection outcome.** Source errors stop this stage after the reports have been saved. Missing auxiliary inputs remain visible but do not invalidate source inspection. A successful inspection does not establish scientific validity or readiness for scoring.

Full provenance notes, references, observed SHA-256 fingerprints, sheet/header declarations, and diagnostics are retained in `results/source_inventory.csv`, `results/table_dimensions.csv`, and `results/supporting_inputs.csv`. These files are regenerated on each run. An observed fingerprint identifies file bytes; it does not independently certify a source version.

In [ ]:
display(inspection.supporting[["input_group", "status"]])
if not inspection.supporting_present:
    print("Supporting inputs are incomplete; see supporting_inputs.csv.")

for label, frame in [("Source files", inspection.sources), ("Source tables", inspection.tables)]:
    failures = frame.loc[frame["status"].ne("OK")]
    if not failures.empty:
        print(label)
        display(failures)

reader_notes = inspection.tables.loc[
    inspection.tables["reader_warnings"].ne(""), ["table_id", "reader_warnings"]
]
if not reader_notes.empty:
    print("Workbook-reader notes")
    display(reader_notes)

inspection.require_sources()
print("Stage 1 source inspection passed. Auxiliary content validation remains pending.")